In [1]:
import importlib
import pickle
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
# import deeptime
import scipy
import pandas as pd
import os
from matplotlib.lines import Line2D
import matplotlib.ticker as mtick
from pygam import LinearGAM, s


import iMSM.extensions.npc.npc
importlib.reload(iMSM.extensions.npc.npc)
from iMSM.extensions.npc.npc import run
# --
import raveh_2025.show_transport_stats_v3
importlib.reload(raveh_2025.show_transport_stats_v3)
from raveh_2025.show_transport_stats_v3 import read_df, calc_filtered_transport_stats
# --
import iMSM.extensions.npc.npc_utils
importlib.reload(iMSM.extensions.npc.npc_utils)
from utils import radius_a_to_kda, amount_to_concentration

In [ ]:
# runs loading for 5-10 time.

for sites in [2,4,6]:
    for radius in [10, 14, 18, 22, 26]:
        print (f"~~~~~~~ Running for {sites} sites and radius {radius} ~~~~~~~")
        for subset in [1.0]: # 
            print (f"~~~~~~~ subset {subset} ~~~~~~~")
    # for radius in [22]:
            
            checkpoint_path = f"data/ntr_variants/{sites}_{radius}_more/"
            params = {}
            params['LOAD_MD_KAP_SITES'] = sites
            params['LOAD_MD_KAP_RADIUS'] = radius
            params['LOAD_MD_KAP_AMOUNT'] = 100
            params['LOAD_MD_BASE_PATH'] = f"/cs/usr/roi.eliasian/LabFolder/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_variants_54R/{sites}_sites_small"
            params['LOAD_MD_SIMS_RANGE'] = range(1, 31)
            params['LOAD_MD_START_TIME_NS'] = 5000
            params['LOAD_MD_END_TIME_NS'] = 10000
            params['LOAD_MD_STEP_NS'] = 100
            params['CUSTOM_FG_COORDS_PATH'] = f"data/ntr_variants/{sites}_{10}_more/2_single_sim_fg_coords/"
            params['TM_PRIOR'] = 0.0001
            params['STATE_CHOICE_METHOD'] = "distance" # options: "prominent", "distance", "nuc_cyt_treshold"
            params['DISTANCE_STATE_THRESHOLD_NM'] = 10
            params['WINDOW_SIZE_STEPS'] = 50
            params['SOFT_SCALE_NM'] = 5
            
            params['MODE'] = "subset"
            params['N_CLUSTERS'] = [320]
            params['DATA_SUBSET'] = subset
            params['DATA_SUBSET_INDEX'] = 0
            params['DATA_SUBSET_MODE'] = "simulation" 

            params['STATE_CHOICE_METHOD'] = "distance_clustering_z" 
            params['CLUSTERING_Z_STRENGTH'] = np.sqrt(458) / 10
            params['N_CLUSTERS'] = [320]
            
            
            run(checkpoint_path, params_override=params, start_stage=1, end_stage=3)

In [8]:
# Runss for various site-radius combinations and sim/time-based data subsets

for sites in [2,4]:
    for radius in [10, 14, 18, 22, 26]:
        print (f"~~~~~~~ Running for {sites} sites and radius {radius} ~~~~~~~")
        for subset in [10/60, 15/60, 20/60, 25/60, 30/60, 40/60, 50/60, 60/60]: # 
            print (f"~~~~~~~ subset {subset} ~~~~~~~")
    # for radius in [22]:
            
            checkpoint_path = f"data/ntr_variants_with_init_time/{sites}_{radius}_more/"
            params = {}
            params['LOAD_MD_KAP_SITES'] = sites
            params['LOAD_MD_KAP_RADIUS'] = radius
            params['LOAD_MD_KAP_AMOUNT'] = 100
            params['LOAD_MD_BASE_PATH'] = f"/cs/usr/roi.eliasian/LabFolder/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_variants/{sites}_sites_small"
            params['LOAD_MD_SIMS_RANGE'] = range(1, 31)
            params['LOAD_MD_START_TIME_NS'] = 5000
            params['LOAD_MD_END_TIME_NS'] = 70000
            params['LOAD_MD_STEP_NS'] = 100
            params['TM_PRIOR'] = 0.0001
            params['STATE_CHOICE_METHOD'] = "distance" # options: "prominent", "distance", "nuc_cyt_treshold"
            params['DISTANCE_STATE_THRESHOLD_NM'] = 10
            params['WINDOW_SIZE_STEPS'] = 50
            params['SOFT_SCALE_NM'] = 5
            
            params['MODE'] = "subset"
            params['N_CLUSTERS'] = [320]
            params['DATA_SUBSET'] = subset
            params['DATA_SUBSET_INDEX'] = 0
            params['DATA_SUBSET_MODE'] = "time" 

            params['STATE_CHOICE_METHOD'] = "distance_clustering_z" 
            params['CLUSTERING_Z_STRENGTH'] = np.sqrt(458) / 10
            params['N_CLUSTERS'] = [320]
            
            params['CUSTOM_FG_COORDS_PATH'] = f"data/ntr_variants/{sites}_{10}_more/2_single_sim_fg_coords/"
            params['CUSTOM_KAP_COORDS_PATH'] = f"data/ntr_variants/{sites}_{radius}_more/1_single_sim_kap_coords/"
            params['CATEGORIZE_SIM_TIMES'] = ["5-10", "10-70"]


            
            
            run(checkpoint_path, params_override=params, start_stage=4, end_stage=8)

~~~~~~~ Running for 2 sites and radius 10 ~~~~~~~
~~~~~~~ subset 0.16666666666666666 ~~~~~~~
Running stage 4...
Stage 4 completed.
Running stage 5...
Using data subset mode: time
Reduced from 320 to 50 clusters after merging heavy nuc/cys centroids (nuc merged: 111, cys merged: 159)
Reduced from 50 to 50 clusters after pruning underpopulated centroids
Stage 5 completed.
Running stage 6...
Stage 6 completed.
Running stage 7...
Permeability for 320 clusters: 707.3789609775105 (units: n_events / s / uM / NPC)
Stage 7 completed.
~~~~~~~ subset 0.25 ~~~~~~~
Running stage 4...
Stage 4 completed.
Running stage 5...
Using data subset mode: time
Reduced from 320 to 50 clusters after merging heavy nuc/cys centroids (nuc merged: 143, cys merged: 127)
Reduced from 50 to 50 clusters after pruning underpopulated centroids
Stage 5 completed.
Running stage 6...
Stage 6 completed.
Running stage 7...
Permeability for 320 clusters: 564.0243893372245 (units: n_events / s / uM / NPC)
Stage 7 completed.
~~~

In [9]:
# Runs the pipeline for low simulation time subsets by proportionally 
# decreasing the window sizes across all combinations (with 5-10 time.)

for sites in [2,4]:
    for radius in [10, 14, 18, 22, 26]:
        print (f"~~~~~~~ Running for {sites} sites and radius {radius} ~~~~~~~")
        for subset, window in [(2/60, 10), (4/60, 20), (8/60, 40)]: # 
            print (f"~~~~~~~ subset {subset} ~~~~~~~")
    # for radius in [22]:
            
            checkpoint_path = f"data/ntr_variants_with_init_time_low_time_subsets/{sites}_{radius}_more/"
            params = {}
            params['LOAD_MD_KAP_SITES'] = sites
            params['LOAD_MD_KAP_RADIUS'] = radius
            params['LOAD_MD_KAP_AMOUNT'] = 100
            params['LOAD_MD_BASE_PATH'] = f"/cs/usr/roi.eliasian/LabFolder/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_variants/{sites}_sites_small"
            params['LOAD_MD_SIMS_RANGE'] = range(1, 31)
            params['LOAD_MD_START_TIME_NS'] = 10000
            params['LOAD_MD_END_TIME_NS'] = 70000
            params['LOAD_MD_STEP_NS'] = 100
            params['CUSTOM_FG_COORDS_PATH'] = f"data/ntr_variants/{sites}_{10}_more/2_single_sim_fg_coords/"
            params['TM_PRIOR'] = 0.0001
            params['STATE_CHOICE_METHOD'] = "distance" # options: "prominent", "distance", "nuc_cyt_treshold"
            params['DISTANCE_STATE_THRESHOLD_NM'] = 10
            params['WINDOW_SIZE_STEPS'] = window
            params['SOFT_SCALE_NM'] = 5
            
            
            params['MODE'] = "subset"
            params['N_CLUSTERS'] = [320]
            params['DATA_SUBSET'] = subset
            params['DATA_SUBSET_INDEX'] = 0
            params['DATA_SUBSET_MODE'] = "time" 

            params['STATE_CHOICE_METHOD'] = "distance_clustering_z" 
            params['CLUSTERING_Z_STRENGTH'] = np.sqrt(458) / 10
            params['N_CLUSTERS'] = [320]
            
            params['CUSTOM_FG_COORDS_PATH'] = f"data/ntr_variants/{sites}_{10}_more/2_single_sim_fg_coords/"
            params['CUSTOM_KAP_COORDS_PATH'] = f"data/ntr_variants/{sites}_{radius}_more/1_single_sim_kap_coords/"
            params['CUSTOM_CATEGORIZED_PATH'] = f"data/ntr_variants_with_init_time/{sites}_{radius}_more/3_categorized.pickle"
            params['CATEGORIZE_SIM_TIMES'] = ["5-10", "10-70"]
            


            
            
            run(checkpoint_path, params_override=params, start_stage=4, end_stage=8)

~~~~~~~ Running for 2 sites and radius 10 ~~~~~~~
~~~~~~~ subset 0.03333333333333333 ~~~~~~~
Running stage 4...
Stage 4 completed.
Running stage 5...
Using data subset mode: time
Reduced from 320 to 50 clusters after merging heavy nuc/cys centroids (nuc merged: 111, cys merged: 159)
Reduced from 50 to 38 clusters after pruning underpopulated centroids
Stage 5 completed.
Running stage 6...
Stage 6 completed.
Running stage 7...
Permeability for 320 clusters: 1258.00111620096 (units: n_events / s / uM / NPC)
Stage 7 completed.
~~~~~~~ subset 0.06666666666666667 ~~~~~~~
Running stage 4...
Stage 4 completed.
Running stage 5...
Using data subset mode: time
Reduced from 320 to 50 clusters after merging heavy nuc/cys centroids (nuc merged: 111, cys merged: 159)
Reduced from 50 to 44 clusters after pruning underpopulated centroids
Stage 5 completed.
Running stage 6...
Stage 6 completed.
Running stage 7...
Permeability for 320 clusters: 818.2660038099627 (units: n_events / s / uM / NPC)
Stage 7 